In [1]:
import pyspark
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from helpers import get_table, get_bronze
print(pyspark.__version__)

3.5.8


In [2]:
builder = (
    SparkSession.builder
    .appName("user_bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

26/04/27 16:11:03 WARN Utils: Your hostname, CVPThuyTTD11-1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 16:11:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thuyttd11/.local/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thuyttd11/.ivy2/cache
The jars for the packages stored in: /home/thuyttd11/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-38ec91bd-ccd1-41e8-bbc0-3230550d8fd2;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 414ms :: artifacts dl 32ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	------------------------

# products silver

+ Read Bronze products data
+ Remove id, ingest_time, source_identifier, batch_id
+ Enforce grain = 1 row per product
+ Standardize column names to snake_case
+ Cast and validate product_id as integer, non-null, unique, and sequential if required
+ Trim and standardize product_name
+ Validate product_name contains exactly 8 distinct valid product names
+ Trim and standardize category to controlled values: Streaming, E-Learning, Cloud Tools, Creative Suite, Productivity
+ Validate category distribution matches business expectation
+ Remove or quarantine rows with invalid or null product_id, product_name, or category
+ De-duplicate records based on product_id
+ Cast and validate created_at as timestamp within the expected 2022 date range
+ Preserve lineage/control columns: ingest_time, source_identifier, batch_id
+ Do not derive churn, usage, or user-related features
+ Do not perform joins
+ Optionally add product_family or business_line for higher-level reporting
+ Write output as Delta table silver_products_clean
+ Perform row-count and null checks against Bronze

## Read Bronze products data

In [3]:
# Bronze layer
products_bronze = "/home/thuyttd11/subscription-analytics/spark-data/bronze/products"
df_bronze_products = get_bronze(products_bronze, spark=spark)
df_bronze_products.show(5)

26/04/27 16:11:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 8:>                                                          (0 + 1) / 1]

+---+----------+------------+-----------+-------------------+--------------------+--------------------+--------------------+
| id|product_id|product_name|   category|         created_at|         ingest_time|   source_identifier|            batch_id|
+---+----------+------------+-----------+-------------------+--------------------+--------------------+--------------------+
|  1|         1|  StreamFlix|  Streaming|2022-05-19 07:00:00|2026-04-22 11:36:...|jdbc:postgresql:/...|app-2026042204353...|
|  2|         2|    BingeNow|  Streaming|2022-08-15 07:00:00|2026-04-22 11:36:...|jdbc:postgresql:/...|app-2026042204353...|
|  3|         3|    LearnPro| E-Learning|2022-05-19 07:00:00|2026-04-22 11:36:...|jdbc:postgresql:/...|app-2026042204353...|
|  4|         4|  SkillBoost| E-Learning|2022-04-24 07:00:00|2026-04-22 11:36:...|jdbc:postgresql:/...|app-2026042204353...|
|  5|         5|   CloudSync|Cloud Tools|2022-04-13 07:00:00|2026-04-22 11:36:...|jdbc:postgresql:/...|app-2026042204353...|


## Remove id, ingest_time, source_identifier, batch_id 

In [4]:
df1 = df_bronze_products.drop("id","source_identifier", "batch_id")
df1.show(5)

+----------+------------+-----------+-------------------+--------------------+
|product_id|product_name|   category|         created_at|         ingest_time|
+----------+------------+-----------+-------------------+--------------------+
|         1|  StreamFlix|  Streaming|2022-05-19 07:00:00|2026-04-22 11:36:...|
|         2|    BingeNow|  Streaming|2022-08-15 07:00:00|2026-04-22 11:36:...|
|         3|    LearnPro| E-Learning|2022-05-19 07:00:00|2026-04-22 11:36:...|
|         4|  SkillBoost| E-Learning|2022-04-24 07:00:00|2026-04-22 11:36:...|
|         5|   CloudSync|Cloud Tools|2022-04-13 07:00:00|2026-04-22 11:36:...|
+----------+------------+-----------+-------------------+--------------------+
only showing top 5 rows



## Enforce grain = 1 row per product

Just to avoid the case that people input the same product with the same created_at multpile times.

Deal with latest products later.

In [5]:
# Remove duplicates based on product_id
# Ensure timestamps
df_clean = (
    df1
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("ingest_time", F.to_timestamp("ingest_time"))
)

# Dedup rule:
# latest created_at
# tie‑breaker: latest ingest_time
w = (
    Window
    .partitionBy("product_id", "created_at")
    .orderBy(
        F.col("ingest_time").desc()
    )
)

df2 = (
    df_clean
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df2_quarantine = (
    df_clean
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") > 1)
    .drop("rn")
)


In [6]:
# Check if any accidental inputs
df2.count(), df2_quarantine.count()

(8, 8)

## Enforce dtypes

In [7]:
df3 = (
    df2
    .withColumn("product_id", col("product_id").cast("int"))
    .withColumn("product_name", col("product_name").cast("string"))
    .withColumn("category", col("category").cast("string"))
    .withColumn("created_at", col("created_at").cast("timestamp"))
    .withColumn("ingest_time", col("ingest_time").cast("timestamp"))
)
df3.dtypes

[('product_id', 'int'),
 ('product_name', 'string'),
 ('category', 'string'),
 ('created_at', 'timestamp'),
 ('ingest_time', 'timestamp')]

## Validate product_id, product_name, category as non-null, unique 

In [8]:
df4 = df3.filter(
    col("product_id").isNotNull() &
    col("product_name").isNotNull() &
    col("category").isNotNull() &
    col("ingest_time").isNotNull()
)
df4_quarantine = df3.filter(
    col("product_id").isNull() |
    col("product_name").isNull() |
    col("category").isNull()     |
    col("ingest_time").isNull()
)
df4.show()
df4_quarantine.show()

+----------+------------+--------------+-------------------+--------------------+
|product_id|product_name|      category|         created_at|         ingest_time|
+----------+------------+--------------+-------------------+--------------------+
|         1|  StreamFlix|     Streaming|2022-05-19 07:00:00|2026-04-22 11:36:...|
|         2|    BingeNow|     Streaming|2022-08-15 07:00:00|2026-04-22 11:36:...|
|         3|    LearnPro|    E-Learning|2022-05-19 07:00:00|2026-04-22 11:36:...|
|         4|  SkillBoost|    E-Learning|2022-04-24 07:00:00|2026-04-22 11:36:...|
|         5|   CloudSync|   Cloud Tools|2022-04-13 07:00:00|2026-04-22 11:36:...|
|         6|    DataFlow|   Cloud Tools|2022-07-27 07:00:00|2026-04-22 11:36:...|
|         7| DesignSuite|Creative Suite|2022-08-11 07:00:00|2026-04-22 11:36:...|
|         8|  TaskMaster|  Productivity|2022-04-26 07:00:00|2026-04-22 11:36:...|
+----------+------------+--------------+-------------------+--------------------+

+----------+---

## Trim product_name only

In [9]:
df5 = df4.withColumn("product_name",F.trim(col("product_name")))
df5.show()

+----------+------------+--------------+-------------------+--------------------+
|product_id|product_name|      category|         created_at|         ingest_time|
+----------+------------+--------------+-------------------+--------------------+
|         1|  StreamFlix|     Streaming|2022-05-19 07:00:00|2026-04-22 11:36:...|
|         2|    BingeNow|     Streaming|2022-08-15 07:00:00|2026-04-22 11:36:...|
|         3|    LearnPro|    E-Learning|2022-05-19 07:00:00|2026-04-22 11:36:...|
|         4|  SkillBoost|    E-Learning|2022-04-24 07:00:00|2026-04-22 11:36:...|
|         5|   CloudSync|   Cloud Tools|2022-04-13 07:00:00|2026-04-22 11:36:...|
|         6|    DataFlow|   Cloud Tools|2022-07-27 07:00:00|2026-04-22 11:36:...|
|         7| DesignSuite|Creative Suite|2022-08-11 07:00:00|2026-04-22 11:36:...|
|         8|  TaskMaster|  Productivity|2022-04-26 07:00:00|2026-04-22 11:36:...|
+----------+------------+--------------+-------------------+--------------------+



## Upsert strategy

In [10]:
# df5.columns
# # Test creating new table


# new_rows = [
#     (1, "Netflix Premium", "Streaming", "2022-01-15 10:00:00"),
#     (102, "Coursera Plus", "E-Learning", "2022-02-10 14:30:00"),
#     (103, "Azure Dev Tools", "Cloud Tools", "2022-03-05 09:15:00"),
#     (104, "Adobe Photoshop", "Creative Suite", "2022-04-20 16:45:00"),
#     (105, "Microsoft 365", "Productivity", "2022-05-12 11:20:00")
# ]

# df_generated = spark.createDataFrame(
#     new_rows,
#     ["product_id", "product_name", "category", "created_at"]
# )
# df_generated.show()
# # Just pass the df_generated in the upsert

In [11]:
df_products_silver = df5

In [14]:
silver_path = "./silver/products"
# The columns that we need
silver_cols = df_products_silver.columns

df_upsert = df_products_silver.select(*silver_cols)

# Keep on ly the row per product_id in the incoming batch
# In this case: keep the latest record by created_at
w = Window.partitionBy("product_id").orderBy(F.col("ingest_time").desc(), F.col("created_at").desc())

df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# If Delta table already exists -> MERGE (upsert)
if DeltaTable.isDeltaTable(spark, silver_path):
    print("Delta table already exists")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (
        delta_target.alias("target")
        .merge(
            df_upsert.alias("source"),
            "target.product_id = source.product_id"
        )
        .whenMatchedUpdate(
            condition="source.ingest_time > target.ingest_time",
            set={c: f"source.{c}" for c in silver_cols}
        )
        .whenNotMatchedInsert(
            values={c: f"source.{c}" for c in silver_cols}
        )
        .execute()
    )

# If Delta table does not exist yet -> initial save
else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )


Delta table does not exist yet


In [15]:
df_user_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_user_silver_read.show(5)

[Stage 57:=============================================>          (41 + 4) / 50]

+----------+------------+-----------+-------------------+--------------------+
|product_id|product_name|   category|         created_at|         ingest_time|
+----------+------------+-----------+-------------------+--------------------+
|         1|  StreamFlix|  Streaming|2022-05-19 07:00:00|2026-04-22 11:36:...|
|         2|    BingeNow|  Streaming|2022-08-15 07:00:00|2026-04-22 11:36:...|
|         3|    LearnPro| E-Learning|2022-05-19 07:00:00|2026-04-22 11:36:...|
|         4|  SkillBoost| E-Learning|2022-04-24 07:00:00|2026-04-22 11:36:...|
|         5|   CloudSync|Cloud Tools|2022-04-13 07:00:00|2026-04-22 11:36:...|
+----------+------------+-----------+-------------------+--------------------+
only showing top 5 rows



In [ ]:
spark.stop()